# Mouse brain: oligodendrocyte and OPC

Two Visium sections share one single-cell reference. `run_all.py` fits each section from `data/mouse_brain/Visium_*.npz`. This notebook reads those fits and shows, for each cell type below, the mixing weight, the program matched across sections by gene-loading correlation, and the top weights.

Change `CELL_TYPES` to tell the same story for another pair. The fit itself is not repeated here.


In [ ]:
from utils.io import load_npz, load_spatial, repo_root
from utils.metrics import corr, match
from utils.plots import gene_weights, panel, tissue
import matplotlib.pyplot as plt
import numpy as np

ROOT = repo_root()
SECTIONS = ["Visium_1", "Visium_2"]
CELL_TYPES = ["Oligodendrocyte", "OPC"]
HIGHLIGHT = {"Oligodendrocyte": ["Plp1", "Mog"], "OPC": ["Pdgfra", "Cspg4"]}
spatial = {name: load_spatial(ROOT / f"data/mouse_brain/{name}.npz") for name in SECTIONS}
fits = {name: load_npz(ROOT / f"Mouse_brain/results/{name}/final.npz") for name in SECTIONS}
genes = spatial["Visium_1"]["genes"]


In [ ]:
def indices(section, cell_type):
    c = list(spatial[section]["cell_types"]).index(cell_type)
    return c, np.flatnonzero(fits[section]["program_cell_type"] == c)

def matched_program(cell_type):
    _, left = indices("Visium_1", cell_type)
    _, right = indices("Visium_2", cell_type)
    i, j, score = max(match(fits["Visium_1"]["loadings"][left], fits["Visium_2"]["loadings"][right]), key=lambda item: item[2])
    return {"Visium_1": int(left[i]), "Visium_2": int(right[j])}, score

def show(cell_type):
    chosen, score = matched_program(cell_type)
    masks = {}
    limit = 1.0
    for section, program in chosen.items():
        c, _ = indices(section, cell_type)
        masks[section] = fits[section]["pi"][:, c] >= 0.05
        limit = max(limit, float(np.max(np.abs(fits[section]["h_mean"][program][masks[section]]))))
    fig = plt.figure(figsize=(7.6, 6.4))
    gs = fig.add_gridspec(3, 4, height_ratios=[1.15, 1.15, 1.05], hspace=.62, wspace=.45)
    letters = iter("abcdefghij")
    for j, section in enumerate(SECTIONS):
        c, _ = indices(section, cell_type)
        coords = spatial[section]["coords"]
        program = chosen[section]
        ax = fig.add_subplot(gs[0, j * 2:j * 2 + 2])
        image = tissue(ax, coords, fits[section]["pi"][:, c], f"{section}: {cell_type} weight", lim=(0, 1), cmap="viridis", invert=True)
        panel(ax, next(letters))
        fig.colorbar(image, ax=ax, orientation="horizontal", fraction=.046, pad=.04, ticks=[0, 0.5, 1])
        ax = fig.add_subplot(gs[1, j * 2:j * 2 + 2])
        image = tissue(ax, coords, fits[section]["h_mean"][program], f"{section}: program activity", mask=masks[section], lim=(-limit, limit), invert=True)
        panel(ax, next(letters))
        fig.colorbar(image, ax=ax, orientation="horizontal", fraction=.046, pad=.04)
        ax = fig.add_subplot(gs[2, j])
        gene_weights(ax, fits[section]["loadings"][program], genes, f"{section}: top genes", highlight=HIGHLIGHT.get(cell_type, ()))
        panel(ax, next(letters))
    ax = fig.add_subplot(gs[2, 2:])
    x = fits["Visium_1"]["loadings"][chosen["Visium_1"]]
    y = fits["Visium_2"]["loadings"][chosen["Visium_2"]]
    ax.scatter(x, y, s=9, color="#82AE9E", linewidths=0)
    hi = max(x.max(), y.max()) * 1.08
    ax.plot([0, hi], [0, hi], color="#B0B0B0", lw=.7)
    ax.set(xlim=(-hi * .02, hi), ylim=(-hi * .02, hi), xlabel="Visium 1 weight", ylabel="Visium 2 weight", title=f"Loading correlation r = {corr(x, y):.3f}")
    panel(ax, next(letters))
    fig.suptitle(cell_type, y=1.01)
    fig.tight_layout()
    return fig, score

for cell_type in CELL_TYPES:
    figure, loading_correlation = show(cell_type)
    print(cell_type, round(loading_correlation, 3))
    figure
